In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from utils.logger import LoggerConfig
from config.io_config import IOConfig
from app.platform_app import PlatformApp
from transformation.gold_loader import *
from transformation.dimensions.dim_date_builder import *

## Set up

In [2]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-10 18:35:43 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-10 18:35:44 | INFO     | Connected to Databricks via Spark Connect.
2026-03-10 18:35:44 | INFO     | Initializing Data Platform...
2026-03-10 18:35:44 | INFO     | Spark session initialized


In [3]:
tables = [
    "fmcg_domain.gold.dim_customers",
    "fmcg_domain.gold.dim_products",
    "fmcg_domain.gold.dim_orders",
    "fmcg_domain.gold.fact_orders",
    "fmcg_domain.gold.dim_gross_price"
]

for table in tables:
    spark.sql(f"DROP TABLE IF EXISTS {table}")

spark.sql("DROP VOLUME IF EXISTS fmcg_domain.gold.checkpoints")

""


## Create catalog fmcg_domain

In [4]:
logger.info("*" * 80)
logger.info("Initializing Unity Catalog and schemas...")
app.create_catalog(name_catalog=app.catalog_name)
for volume in app.DEFAULT_VOLUMES:
    app.create_volume(schema_name="bronze", volume_name=volume)
    app.create_volume(schema_name="gold", volume_name=volume)
logger.info("Catalog and schemas initialized successfully.")
logger.info("*" * 80)

2026-03-10 18:35:57 | INFO     | ********************************************************************************
2026-03-10 18:35:57 | INFO     | Initializing Unity Catalog and schemas...
2026-03-10 18:35:57 | INFO     | Creating catalog: fmcg_domain
2026-03-10 18:35:58 | INFO     | CATALOG fmcg_domain created successfully.
2026-03-10 18:35:58 | INFO     | Creating schema: bronze
2026-03-10 18:35:59 | INFO     | SCHEMA bronze created successfully.
2026-03-10 18:35:59 | INFO     | Creating schema: silver
2026-03-10 18:36:00 | INFO     | SCHEMA silver created successfully.
2026-03-10 18:36:00 | INFO     | Creating schema: gold
2026-03-10 18:36:01 | INFO     | SCHEMA gold created successfully.
2026-03-10 18:36:01 | INFO     | Catalog setup completed.
2026-03-10 18:36:01 | INFO     | Creating volume: fmcg_domain.bronze.checkpoints
2026-03-10 18:36:01 | INFO     | Volume checkpoints created successfully.
2026-03-10 18:36:01 | INFO     | Creating volume: fmcg_domain.gold.checkpoints
2026-03

## Source Data Upload

In [5]:
logger.info("*" * 80)
logger.info("Uploading source datasets to staging volume...")
app.upload_local_to_uc_volume(local_base=IOConfig.DATASET_DIR, catalog=app.catalog_name,
                              schema="source", volume="source_data")
logger.info("Source data upload completed.")
logger.info("*" * 80)

2026-03-10 18:36:02 | INFO     | ********************************************************************************
2026-03-10 18:36:02 | INFO     | Uploading source datasets to staging volume...
2026-03-10 18:36:03 | INFO     | Creating schema: source
2026-03-10 18:36:04 | INFO     | Starting upload to Unity Catalog Volume...
2026-03-10 18:36:05 | INFO     | Skip (unchanged): /Volumes/fmcg_domain/source/source_data/customers/dim_customers.csv
2026-03-10 18:36:05 | INFO     | Skip (unchanged): /Volumes/fmcg_domain/source/source_data/gross_price/dim_gross_price.csv
2026-03-10 18:36:06 | INFO     | Skip (unchanged): /Volumes/fmcg_domain/source/source_data/orders/fact_orders.csv
2026-03-10 18:36:06 | INFO     | Skip (unchanged): /Volumes/fmcg_domain/source/source_data/products/dim_products.csv
2026-03-10 18:36:06 | INFO     | Source data upload completed.
2026-03-10 18:36:06 | INFO     | ********************************************************************************


## Load data into Gold Layer

In [6]:
logger.info("Ingesting data into Gold layer...")
gold_loader(spark=spark, logger=logger, list_entity=app.DEFAULT_ENTITIES,
            path_data=IOConfig.COMMON, path_cp=IOConfig.CP_PATH_GOLD,
            name_catalog=app.catalog_name)

2026-03-10 18:36:06 | INFO     | Ingesting data into Gold layer...
2026-03-10 18:36:06 | INFO     | Starting Gold ingestion for entity: customers
2026-03-10 18:36:06 | INFO     | Uploading file: /Volumes/fmcg_domain/source/source_data/customers
2026-03-10 18:36:18 | INFO     | Customers: Schema inferred successfully
2026-03-10 18:36:29 | INFO     | fmcg_domain.gold.dim_customers: ingestion completed
+-------------+--------------------+------+--------------+--------+
|customer_code|customer            |market|platform      |channel |
+-------------+--------------------+------+--------------+--------+
|70002017     |Sporty Essentials   |India |Brick & Mortar|Retailer|
|70002018     |PlayMore Sports     |India |Brick & Mortar|Retailer|
|90002001     |SportsHub           |India |E-Commerce    |Retailer|
|90002002     |PowerPlay Sports    |India |Brick & Mortar|Retailer|
|90002003     |Winning Gear        |India |Brick & Mortar|Retailer|
|90002004     |Atlikon Sports India|India |Brick & Mo

## Create table dim_date for gold layer

In [8]:
# start date and end date
start_date = "2024-01-01"
end_date   = "2025-12-01"

create_dim_date(spark=spark, start_date=start_date, end_date=end_date,
                table_name="fmcg_domain.gold.dim_date", logger=logger)

2026-03-10 18:37:09 | INFO     | Generating dim_date from 2024-01-01 to 2025-12-01
2026-03-10 18:37:10 | INFO     | Calendar attributes successfully generated
2026-03-10 18:37:13 | INFO     | dim_date successfully written to table: fmcg_domain.gold.dim_date
